<a href="https://colab.research.google.com/github/EvenSol/NeqSim-Colab/blob/master/notebooks/AI/agentic_neqsim_workflow_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agentic NeqSim Workflow Demo

Show how a script or AI agent can define a process case, run NeqSim, collect validation evidence, and write a small machine-readable result object.


## Setup

Install imports and define reusable functions.


In [ ]:
# Install NeqSim when running in a fresh Colab session.
try:
    import neqsim
except ImportError:
    %pip install neqsim

import json
import pandas as pd
import matplotlib.pyplot as plt
try:
    from neqsim import jneqsim
except ImportError:
    from neqsim import jNeqSim as jneqsim
from neqsim.thermo import TPflash

SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
Stream = jneqsim.process.equipment.stream.Stream
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

def make_gas(temperature_c=25.0, pressure_bara=60.0):
    fluid = SystemSrkEos(273.15 + temperature_c, pressure_bara)
    fluid.addComponent("nitrogen", 0.01)
    fluid.addComponent("CO2", 0.02)
    fluid.addComponent("methane", 0.86)
    fluid.addComponent("ethane", 0.07)
    fluid.addComponent("propane", 0.03)
    fluid.addComponent("n-butane", 0.01)
    fluid.setMixingRule("classic")
    TPflash(fluid)
    fluid.initProperties()
    return fluid

## Define the Case as Data

A structured case dictionary makes the workflow easy to reproduce and modify.


In [ ]:
case = {
    "feed_temperature_C": 25.0,
    "feed_pressure_bara": 55.0,
    "feed_flow_kg_per_hr": 9000.0,
    "compressor_outlet_pressure_bara": 105.0,
}
case


## Build and Run from the Case

The agent-style pattern keeps inputs, model execution, and outputs separate.


In [ ]:
Compressor = jneqsim.process.equipment.compressor.Compressor
fluid = make_gas(case["feed_temperature_C"], case["feed_pressure_bara"])
feed = Stream("Feed gas", fluid)
feed.setFlowRate(case["feed_flow_kg_per_hr"], "kg/hr")
compressor = Compressor("Compressor", feed)
compressor.setOutletPressure(case["compressor_outlet_pressure_bara"])
process = ProcessSystem()
process.add(feed)
process.add(compressor)
process.run()
results = {
    "inputs": case,
    "key_results": {
        "compressor_power_kW": compressor.getPower("kW"),
        "outlet_temperature_C": compressor.getOutStream().getTemperature("C"),
    },
    "validation": {
        "ran_without_exception": True,
        "positive_power": compressor.getPower("kW") > 0.0,
    },
}
print(json.dumps(results, indent=2))


## Use the Result Object

A downstream agent can plot, compare, or report from the same JSON-style object.


In [ ]:
pd.DataFrame([results["key_results"]])
